# Load fire data into Tiger Data (TimescaleDB)
Loads the cleaned FIRMS detections and the fire table into a Tiger Cloud service, builds two continuous
aggregates, and tests the three queries the website's API routes will use.

- The connection string comes from `TIGER_DATABASE_URL` in the repo-root `.env`. It is never printed.
- Without it, the data-prep cells still run and every database cell prints "No TIGER_DATABASE_URL; skipped".
- The database cells are idempotent: they drop and recreate everything, so re-running is safe.

## Config

In [ ]:
from pathlib import Path
import io
import os
import sys
import time
import numpy as np
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv
from IPython.display import display

folder = Path.cwd().resolve()
REPO_ROOT = next((p for p in (folder, *folder.parents) if (p / "CLAUDE.md").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Launch from inside the repository.")

DRY_RUN = False            # True: load only the first DRY_RUN_ROWS detections. False: load all of them.
DRY_RUN_ROWS = 10_000
GRID_DEG = 0.1             # map cell size in degrees
YEARS = [2019, 2020, 2021, 2022, 2023]
COPY_CHUNK_ROWS = 50_000
EXAMPLE_START, EXAMPLE_END = "2023-07-01", "2023-09-30"   # inclusive dates for the test queries

FIRMS_DIR = REPO_ROOT / "data" / "processed" / "firms"
IGNITIONS_PATH = REPO_ROOT / "data" / "processed" / "ignitions.parquet"
FIRE_CENTRES_PATH = REPO_ROOT / "data" / "reference" / "bc_fire_centres.geojson"
MODEL_DIR = REPO_ROOT / "data" / "processed" / "model"
QUERIES_PATH = REPO_ROOT / "notebooks" / "tiger_queries.sql"

load_dotenv(REPO_ROOT / ".env", override=False)
DATABASE_URL = os.getenv("TIGER_DATABASE_URL", "").strip()
HAS_DATABASE = bool(DATABASE_URL)
SKIPPED = "No TIGER_DATABASE_URL; skipped"
print(f"DRY_RUN = {DRY_RUN}; GRID_DEG = {GRID_DEG}; database configured: {HAS_DATABASE}")

DRY_RUN = True; GRID_DEG = 0.1; database configured: False


## 1. Detections
The five cleaned FIRMS files, vegetation fires only (`type == 0`). Each detection gets a fire centre
(`region`) and a map cell (`cell_id`, from flooring latitude and longitude to `GRID_DEG`).

In [2]:
if not FIRE_CENTRES_PATH.exists():
    raise FileNotFoundError(f"{FIRE_CENTRES_PATH.relative_to(REPO_ROOT)} is missing; run 04_candidates_costs first.")
fire_centres = gpd.read_file(FIRE_CENTRES_PATH)
fire_centres["region"] = fire_centres["MOF_FIRE_CENTRE_NAME"].str.replace(" Fire Centre", "", regex=False)
fire_centres = fire_centres[["region", "geometry"]].to_crs("EPSG:3005")


def assign_region(frame, label):
    """Fire centre containing each point, or the nearest one if it falls just outside (as in notebook 04)."""
    points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(frame["lon"], frame["lat"]), crs="EPSG:4326",
                              index=frame.index).to_crs("EPSG:3005")
    inside = gpd.sjoin(points, fire_centres, how="left", predicate="within")
    region = inside.loc[~inside.index.duplicated(), "region"].reindex(frame.index)
    outside = region.isna()
    if outside.any():
        nearest = gpd.sjoin_nearest(points[outside], fire_centres, how="left", distance_col="distance_m")
        nearest = nearest[~nearest.index.duplicated()]
        region.loc[outside] = nearest["region"]
        print(f"{label}: {outside.sum():,} of {len(frame):,} outside every polygon "
              f"(farthest {nearest['distance_m'].max() / 1000:.1f} km), assigned to the nearest centre")
    else:
        print(f"{label}: all {len(frame):,} inside a fire centre")
    return region.to_numpy()


def cell_ids(lat, lon):
    """Cell label such as "49.2_-123.1": the south-west corner of the GRID_DEG cell."""
    decimals = max(0, int(np.ceil(-np.log10(GRID_DEG))))
    corner = lambda values: (np.floor(values / GRID_DEG) * GRID_DEG).round(decimals) + 0.0   # + 0.0 turns -0.0 into 0.0
    return (pd.Series(corner(lat)).map(f"{{:.{decimals}f}}".format) + "_"
            + pd.Series(corner(lon)).map(f"{{:.{decimals}f}}".format)).to_numpy()


started = time.perf_counter()
frames = []
for year in YEARS:
    frame = pd.read_csv(FIRMS_DIR / f"firms_bc_{year}.csv", low_memory=False,
                        usecols=["latitude", "longitude", "frp", "confidence", "daynight", "type", "source", "acq_datetime"])
    frames.append(frame[frame["type"] == 0])
raw = pd.concat(frames, ignore_index=True)
detections = pd.DataFrame({
    "detected_at": pd.to_datetime(raw["acq_datetime"], utc=True),
    "lat": raw["latitude"].astype(float), "lon": raw["longitude"].astype(float), "frp": raw["frp"].astype(float),
    "confidence": raw["confidence"].astype(str), "daynight": raw["daynight"].astype(str), "source": raw["source"].astype(str),
})
detections.insert(1, "year", detections["detected_at"].dt.year.astype(int))
detections = detections.sort_values("detected_at", kind="stable").reset_index(drop=True)
detections["region"] = assign_region(detections, "detections")
detections["cell_id"] = cell_ids(detections["lat"].to_numpy(), detections["lon"].to_numpy())
print(f"{len(detections):,} detections prepared in {time.perf_counter() - started:.0f}s; "
      f"{detections['cell_id'].nunique():,} distinct {GRID_DEG}-degree cells")
display(detections.head(3))

detections: 541 of 538,508 outside every polygon (farthest 0.9 km), assigned to the nearest centre


538,508 detections prepared in 8s; 4,288 distinct 0.1-degree cells


,detected_at,year,lat,lon,frp,confidence,daynight,source,region,cell_id
0,2019-05-01 10:41:00+00:00,2019,53.99883,-128.69853,0.43,n,N,VIIRS_SNPP_SP,Northwest,53.9_-128.7
1,2019-05-01 10:41:00+00:00,2019,54.02320,-128.68787,0.61,n,N,VIIRS_SNPP_SP,Northwest,54.0_-128.7
2,2019-05-01 10:41:00+00:00,2019,54.02480,-128.67470,0.94,n,N,VIIRS_SNPP_SP,Northwest,54.0_-128.7


## 2. Ignitions
All fires from the fire table, static suspects included (the website filters them out in its query).

In [3]:
fire_table = pd.read_parquet(IGNITIONS_PATH)
ignitions = fire_table[["fire_id", "year", "ignition_time_utc", "lat", "lon", "weight", "active_days_capped",
                        "static_suspect"]].copy()
ignitions["fire_id"] = ignitions["fire_id"].astype(str)
assert (ignitions["weight"] % 1 == 0).all()
ignitions["weight"] = ignitions["weight"].astype(int)
ignitions["region"] = assign_region(ignitions, "ignitions")
print(f"{len(ignitions):,} ignitions ({ignitions['static_suspect'].sum()} static suspects)")
display(ignitions.head(3))

ignitions: 13 of 7,227 outside every polygon (farthest 0.8 km), assigned to the nearest centre
7,227 ignitions (195 static suspects)


,fire_id,year,ignition_time_utc,lat,lon,weight,active_days_capped,static_suspect,region
0,2019-00001,2019,2019-05-01 10:41:00+00:00,54.01561,-128.687033,3,5,False,Northwest
1,2019-00002,2019,2019-05-02 10:23:00+00:00,50.46657,-126.280820,1,1,False,Coastal
2,2019-00003,2019,2019-05-03 10:04:00+00:00,50.98555,-118.272400,1,1,False,Southeast


## 3. Checks

In [4]:
DETECTION_COLUMNS = ["detected_at", "year", "lat", "lon", "frp", "confidence", "daynight", "source", "region", "cell_id"]
IGNITION_COLUMNS = ["fire_id", "year", "ignition_time_utc", "lat", "lon", "weight", "active_days_capped",
                    "static_suspect", "region"]
assert list(detections.columns) == DETECTION_COLUMNS and list(ignitions.columns) == IGNITION_COLUMNS

checks = {
    "detections: no missing values": not detections.isna().any().any(),
    "ignitions: no missing values": not ignitions.isna().any().any(),
    "detections: timestamps are UTC and inside 2019-2023": str(detections["detected_at"].dt.tz) == "UTC"
        and detections["year"].isin(YEARS).all(),
    "detections: confidence is n or h, daynight is D or N": detections["confidence"].isin(["n", "h"]).all()
        and detections["daynight"].isin(["D", "N"]).all(),
    "detections: sorted by time": detections["detected_at"].is_monotonic_increasing,
    "ignitions: fire_id is unique": ignitions["fire_id"].is_unique,
    "ignitions: 7,227 rows": len(ignitions) == 7227,
    "six fire centres in both tables": detections["region"].nunique() == 6 and set(ignitions["region"]) == set(detections["region"]),
}
if (MODEL_DIR / "demand_train.parquet").exists():
    # The same fires were assigned a region in notebook 04; the two assignments must agree.
    modelled = pd.concat([pd.read_parquet(MODEL_DIR / f"demand_{split}.parquet")[["fire_id", "region"]] for split in ("train", "test")])
    merged = modelled.merge(ignitions[["fire_id", "region"]], on="fire_id", suffixes=("_04", "_here"))
    checks[f"ignition regions match notebook 04 ({len(merged):,} fires)"] = (
        len(merged) == len(modelled) and (merged["region_04"] == merged["region_here"]).all())
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
if not all(checks.values()):
    raise ValueError("A data check failed; nothing will be loaded.")

print(f"\nDetections: {len(detections):,} rows, {detections['detected_at'].min()} to {detections['detected_at'].max()}")
display(detections.groupby("year").agg(detections=("lat", "size"), frp_sum=("frp", "sum"),
                                       noaa20_fill=("source", lambda s: int((s == "VIIRS_NOAA20_SP").sum()))).round(0))
display(detections.groupby("region").size().rename("detections").to_frame().join(
    ignitions.groupby("region").size().rename("ignitions")))

detections_to_load = detections.head(DRY_RUN_ROWS) if DRY_RUN else detections
print(f"Rows that will be loaded: {len(detections_to_load):,} detections "
      f"({'DRY_RUN: first ' + format(DRY_RUN_ROWS, ',') if DRY_RUN else 'all'}), {len(ignitions):,} ignitions")

PASS  detections: no missing values
PASS  ignitions: no missing values
PASS  detections: timestamps are UTC and inside 2019-2023
PASS  detections: confidence is n or h, daynight is D or N
PASS  detections: sorted by time
PASS  ignitions: fire_id is unique
PASS  ignitions: 7,227 rows
PASS  six fire centres in both tables
PASS  ignition regions match notebook 04 (7,032 fires)

Detections: 538,508 rows, 2019-05-01 10:41:00+00:00 to 2023-10-31 21:48:00+00:00


,detections,frp_sum,noaa20_fill
year,,,
2019,11185,138101.0,0
2020,6605,73750.0,0
2021,128131,1638782.0,0
2022,26930,327312.0,2632
2023,365657,4576038.0,0


,detections,ignitions
region,,
Cariboo,27752,502
Coastal,21352,1023
Kamloops,104449,1020
Northwest,41981,943
Prince George,308121,2730
Southeast,34853,1009


Rows that will be loaded: 10,000 detections (DRY_RUN: first 10,000), 7,227 ignitions


## 4. The website's queries
The three statements the API routes will use, with `%(start)s` and `%(end)s` parameters (inclusive dates,
`YYYY-MM-DD`, UTC). They are saved to `notebooks/tiger_queries.sql` so the website can reuse them.
This cell needs no database.

In [5]:
# Dates are inclusive and interpreted as UTC days, whatever the session time zone is.
DATE_RANGE = """{column} >= (%(start)s::date)::timestamp AT TIME ZONE 'UTC'
  AND {column} < (%(end)s::date + 1)::timestamp AT TIME ZONE 'UTC'"""

QUERIES = {
    "daily_detections_by_region": f"""
SELECT day, region, detections, frp_sum
FROM detections_daily_region
WHERE {DATE_RANGE.format(column="day")}
ORDER BY day, region;""",
    "cell_totals": f"""
SELECT cell_id,
       sum(detections) AS detections,
       sum(frp_sum) AS frp_sum,
       sum(lat_avg * detections) / sum(detections) AS lat,
       sum(lon_avg * detections) / sum(detections) AS lon
FROM detections_daily_cell
WHERE {DATE_RANGE.format(column="day")}
GROUP BY cell_id
ORDER BY detections DESC;""",
    "ignitions_between": f"""
SELECT fire_id, year, ignition_time_utc, lat, lon, weight, active_days_capped, region
FROM ignitions
WHERE NOT static_suspect
  AND {DATE_RANGE.format(column="ignition_time_utc")}
ORDER BY ignition_time_utc, fire_id;""",
}
QUERY_NOTES = {
    "daily_detections_by_region": "Daily detections per fire centre between two dates.",
    "cell_totals": "Totals per map cell for a date range (map heat layer). lat/lon are the detection-weighted centre.",
    "ignitions_between": "Ignitions between two dates, excluding static heat-source suspects.",
}
sql_file = ["-- Queries used by the website's API routes. Written by notebooks/06b_export_tiger.ipynb; do not edit by hand.",
            "-- Parameters: %(start)s and %(end)s are inclusive UTC dates, 'YYYY-MM-DD' (psycopg named-parameter style).",
            f"-- Map cells are {GRID_DEG} degrees; cell_id is the south-west corner, for example '49.2_-123.1'.", ""]
for name, statement in QUERIES.items():
    sql_file += [f"-- name: {name}", f"-- {QUERY_NOTES[name]}", statement.strip(), ""]
QUERIES_PATH.write_text("\n".join(sql_file), encoding="utf-8", newline="\n")
print(f"Saved {len(QUERIES)} queries to {QUERIES_PATH.relative_to(REPO_ROOT)}")

Saved 3 queries to notebooks\tiger_queries.sql


## 5. Connect
From here on every cell needs the database. The connection uses autocommit, because continuous aggregates
cannot be created or refreshed inside a transaction.

In [6]:
connection = None
if not HAS_DATABASE:
    print(SKIPPED)
else:
    import psycopg
    from psycopg.conninfo import conninfo_to_dict

    def without_secrets(message):
        """Remove the password and the full connection string from an error message."""
        message = str(message).replace(DATABASE_URL, "[connection string]")
        password = conninfo_to_dict(DATABASE_URL).get("password")
        return message.replace(password, "[password]") if password else message

    try:
        connection = psycopg.connect(DATABASE_URL, autocommit=True, connect_timeout=30)
    except psycopg.Error as error:
        raise RuntimeError(f"Could not connect to Tiger: {without_secrets(error)}") from None
    connection.execute("SET TIME ZONE 'UTC'")
    connection.execute("CREATE EXTENSION IF NOT EXISTS timescaledb")
    version = connection.execute("SELECT extversion FROM pg_extension WHERE extname = 'timescaledb'").fetchone()
    print(f"Connected. PostgreSQL {connection.info.server_version}, TimescaleDB {version[0] if version else 'not installed'}")

No TIGER_DATABASE_URL; skipped


## 6. Tables
Drop what exists, then create `detections` as a hypertable on `detected_at` and `ignitions` as a regular table.

In [7]:
SCHEMA_SQL = [
    "DROP MATERIALIZED VIEW IF EXISTS detections_daily_region CASCADE",
    "DROP MATERIALIZED VIEW IF EXISTS detections_daily_cell CASCADE",
    "DROP TABLE IF EXISTS detections CASCADE",
    "DROP TABLE IF EXISTS ignitions CASCADE",
    """CREATE TABLE detections (
        detected_at TIMESTAMPTZ NOT NULL,
        year SMALLINT NOT NULL,
        lat DOUBLE PRECISION NOT NULL,
        lon DOUBLE PRECISION NOT NULL,
        frp DOUBLE PRECISION NOT NULL,
        confidence TEXT NOT NULL,
        daynight TEXT NOT NULL,
        source TEXT NOT NULL,
        region TEXT NOT NULL,
        cell_id TEXT NOT NULL
    )""",
    "SELECT create_hypertable('detections', 'detected_at', if_not_exists => TRUE)",
    "CREATE INDEX IF NOT EXISTS detections_region_time_idx ON detections (region, detected_at)",
    """CREATE TABLE ignitions (
        fire_id TEXT PRIMARY KEY,
        year SMALLINT NOT NULL,
        ignition_time_utc TIMESTAMPTZ NOT NULL,
        lat DOUBLE PRECISION NOT NULL,
        lon DOUBLE PRECISION NOT NULL,
        weight SMALLINT NOT NULL,
        active_days_capped SMALLINT NOT NULL,
        static_suspect BOOLEAN NOT NULL,
        region TEXT NOT NULL
    )""",
    "CREATE INDEX IF NOT EXISTS ignitions_time_idx ON ignitions (ignition_time_utc)",
]
if connection is None:
    print(SKIPPED)
else:
    for statement in SCHEMA_SQL:
        connection.execute(statement)
    print("Created detections (hypertable on detected_at) and ignitions, with their indexes.")

No TIGER_DATABASE_URL; skipped


## 7. Bulk load
`COPY ... FROM STDIN` in CSV chunks. With `DRY_RUN = True` only the first 10,000 detections are loaded.

In [8]:
def copy_table(table, frame, columns):
    """Load a DataFrame with COPY; returns (rows loaded, seconds)."""
    started = time.perf_counter()
    statement = f"COPY {table} ({', '.join(columns)}) FROM STDIN WITH (FORMAT csv)"
    with connection.cursor() as cursor, cursor.copy(statement) as copy:
        for offset in range(0, len(frame), COPY_CHUNK_ROWS):
            buffer = io.StringIO()
            frame.iloc[offset:offset + COPY_CHUNK_ROWS].to_csv(buffer, columns=columns, index=False, header=False)
            copy.write(buffer.getvalue())
    loaded = connection.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    return loaded, time.perf_counter() - started


if connection is None:
    print(SKIPPED)
else:
    for table, frame, columns in [("detections", detections_to_load, DETECTION_COLUMNS), ("ignitions", ignitions, IGNITION_COLUMNS)]:
        loaded, seconds = copy_table(table, frame, columns)
        print(f"{table}: {loaded:,} rows loaded in {seconds:.1f}s")
        if loaded != len(frame):
            raise RuntimeError(f"{table}: expected {len(frame):,} rows, found {loaded:,}")

No TIGER_DATABASE_URL; skipped


## 8. Continuous aggregates
Daily roll-ups that Tiger keeps materialized: per fire centre, and per map cell (with the average position
for drawing the cell). Both are created `WITH NO DATA` and then refreshed over the whole table.

In [9]:
AGGREGATE_SQL = [
    """CREATE MATERIALIZED VIEW detections_daily_region WITH (timescaledb.continuous) AS
       SELECT time_bucket('1 day', detected_at) AS day, region, count(*) AS detections, sum(frp) AS frp_sum
       FROM detections
       GROUP BY time_bucket('1 day', detected_at), region
       WITH NO DATA""",
    """CREATE MATERIALIZED VIEW detections_daily_cell WITH (timescaledb.continuous) AS
       SELECT time_bucket('1 day', detected_at) AS day, cell_id, count(*) AS detections, sum(frp) AS frp_sum,
              avg(lat) AS lat_avg, avg(lon) AS lon_avg
       FROM detections
       GROUP BY time_bucket('1 day', detected_at), cell_id
       WITH NO DATA""",
]
if connection is None:
    print(SKIPPED)
else:
    for statement in AGGREGATE_SQL:
        connection.execute(statement)
    for view in ("detections_daily_region", "detections_daily_cell"):
        started = time.perf_counter()
        connection.execute(f"CALL refresh_continuous_aggregate('{view}', NULL, NULL)")
        rows = connection.execute(f"SELECT count(*) FROM {view}").fetchone()[0]
        print(f"{view}: refreshed in {time.perf_counter() - started:.1f}s, {rows:,} rows")

No TIGER_DATABASE_URL; skipped


## 9. Test the website's queries
Runs each saved query and prints the row count, timing and first rows. With `DRY_RUN = True` only early-2019
detections are loaded, so the detection queries use the loaded date range instead of the 2023 example.

In [10]:
if connection is None:
    print(SKIPPED)
else:
    if DRY_RUN:
        detection_range = {"start": str(detections_to_load["detected_at"].min().date()),
                           "end": str(detections_to_load["detected_at"].max().date())}
    else:
        detection_range = {"start": EXAMPLE_START, "end": EXAMPLE_END}
    parameters = {"daily_detections_by_region": detection_range, "cell_totals": detection_range,
                  "ignitions_between": {"start": EXAMPLE_START, "end": EXAMPLE_END}}
    for name, statement in QUERIES.items():
        started = time.perf_counter()
        cursor = connection.execute(statement, parameters[name])
        rows = cursor.fetchall()
        milliseconds = 1000 * (time.perf_counter() - started)
        print(f"{name} {parameters[name]}: {len(rows):,} rows in {milliseconds:.0f} ms")
        display(pd.DataFrame(rows, columns=[column.name for column in cursor.description]).head(5))

    # Cross-check the first query against pandas on the rows that were loaded.
    loaded = detections_to_load[detections_to_load["detected_at"].dt.date.astype(str).between(detection_range["start"], detection_range["end"])]
    in_database = connection.execute(
        "SELECT coalesce(sum(detections), 0) FROM detections_daily_region WHERE "
        + DATE_RANGE.format(column="day"), detection_range).fetchone()[0]
    print(f"Cross-check: {int(in_database):,} detections in the aggregate, {len(loaded):,} in pandas for the same dates")
    if int(in_database) != len(loaded):
        raise RuntimeError("The continuous aggregate does not match the loaded detections.")

No TIGER_DATABASE_URL; skipped


## 10. Close

In [11]:
if connection is None:
    print(SKIPPED)
else:
    connection.close()
    print("Connection closed.")

No TIGER_DATABASE_URL; skipped
